# Final project: infectious_cases

Dataset with infectious disease stats by country and year. Steps from the task:
1. Load data into schema `pandemic`
2. Normalize the table to 3NF
3. Analyse `Number_rabies`
4. Column with difference in years
5. Own functions

## 1. Load data

In [1]:
%pip install -q pgserver psycopg2-binary sqlalchemy pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import platform
from urllib.request import urlretrieve
import pandas as pd
import pgserver
from sqlalchemy import create_engine, text

pg = pgserver.get_server('/tmp/fp_pg', cleanup_mode='stop')
TZ = "GMT" if platform.system() == "Windows" else "UTC" # both are +00
engine = create_engine(pg.get_uri(), future=True, connect_args={"options": f"-c timezone={TZ}"})

with engine.connect() as conn:
    print(conn.execute(text('SELECT version()')).scalar_one())

PostgreSQL 16.2 on x86_64-pc-mingw64, compiled by gcc.exe (x86_64-posix-seh-rev2, Built by MinGW-W64 project) 12.2.0, 64-bit


In [3]:
def run_sql(sql):
    # for CREATE / INSERT / DROP (no result)
    with engine.begin() as conn:
        conn.execute(text(sql))

def show(sql):
    # for SELECT, returns a table
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn)

### Schema pandemic

`SET search_path` works only for one session, and here every `run_sql` / `show` opens a new conection.
So I also set it on the database with `ALTER DATABASE`, then all new connections use `pandemic` by default.

In [4]:
run_sql('''
CREATE SCHEMA IF NOT EXISTS pandemic;
SET search_path TO pandemic;
ALTER DATABASE postgres SET search_path TO pandemic;
''')
engine.dispose() # drop old connections, new ones get the new search_path

show('SHOW search_path')

,search_path
0,pandemic


### Import CSV

The file goes to `data/`, the cell downloads it only if it's not there yet.
All columns are loaded as text like in the CSV, empty cells stay empty strings `''` (not NULL).
Column names are made lower case so I dont need quotes in SQL.

In [5]:
URL = 'https://raw.githubusercontent.com/dvankevich/goit-rdb-fp/HEAD/infectious_cases.csv'
csv_path = 'data/infectious_cases.csv'

if not os.path.exists(csv_path):
    os.makedirs('data', exist_ok=True)
    urlretrieve(URL, csv_path)

df = pd.read_csv(csv_path, dtype=str, keep_default_na=False)
df.columns = df.columns.str.lower()

run_sql('DROP TABLE IF EXISTS infectious_cases_norm; DROP TABLE IF EXISTS entities;')
df.to_sql('infectious_cases', engine, schema='pandemic', if_exists='replace', index=False, chunksize=10_000)

show('SELECT COUNT(*) AS n_rows FROM infectious_cases')

,n_rows
0,10521


### Look at the data

In [6]:
pd.set_option('display.max_columns', None)

show('SELECT * FROM infectious_cases LIMIT 10')

,entity,code,year,number_yaws,polio_cases,cases_guinea_worm,number_rabies,number_malaria,number_hiv,number_tuberculosis,number_smallpox,number_cholera_cases
0,Afghanistan,AFG,1980,,6160,0,,,,,,
1,Afghanistan,AFG,1981,,5859,0,,,,,,
2,Afghanistan,AFG,1982,,9730,0,,,,,,
3,Afghanistan,AFG,1983,,13937,0,,,,,,
4,Afghanistan,AFG,1984,,3864,0,,,,,,
5,Afghanistan,AFG,1985,,13867,0,,,,,,
6,Afghanistan,AFG,1986,,12901,0,,,,,,
7,Afghanistan,AFG,1987,,4396,0,,,,,,
8,Afghanistan,AFG,1988,,2149,0,,,,,,
9,Afghanistan,AFG,1989,,385,0,,,,,,


In [7]:
show('''
SELECT column_name, data_type
FROM information_schema.columns
WHERE table_schema = 'pandemic' AND table_name = 'infectious_cases'
ORDER BY ordinal_position;
''')

,column_name,data_type
0,entity,text
1,code,text
2,year,text
3,number_yaws,text
4,polio_cases,text
5,cases_guinea_worm,text
6,number_rabies,text
7,number_malaria,text
8,number_hiv,text
9,number_tuberculosis,text


In [8]:
show('''
SELECT COUNT(DISTINCT entity) AS n_entities,
    COUNT(DISTINCT code) AS n_codes,
    MIN(year) AS min_year,
    MAX(year) AS max_year,
    COUNT(*) FILTER (WHERE code = '') AS rows_without_code,
    COUNT(*) FILTER (WHERE number_rabies = '') AS rows_without_rabies
FROM infectious_cases;
''')

,n_entities,n_codes,min_year,max_year,rows_without_code,rows_without_rabies
0,245,212,1920,2023,1128,3741


10 521 rows, 245 entities, 212 codes. Years from 1920 to 2023 (min/max here are text, but all years have 4 digits so its ok).
Some entities are not countries (`Africa`, `G20`, WHO and WB regions...), they have empty `code`, 1 128 rows like this.
`number_rabies` is empty in 3 741 rows. Numbers are not always integers, rabies has values like `2.145772`.

## 2. Normalization to 3NF

In `infectious_cases` the key is `(entity, year)`. But `code` depends only on `entity`, not on the year,
so its repeated in every row of the same country. So I move `entity` + `code` into own table `entities`
and in the second table keep only `entity_id`.

- `entities` - one row per unique `entity` + `code`, `id` is the key
- `infectious_cases_norm` - `entity_id` (FK), `year` and all disease numbers

Empty strings become NULL and numbers get a real type (`NUMERIC`), year is `INTEGER`.

In [9]:
run_sql('''
DROP TABLE IF EXISTS infectious_cases_norm;
DROP TABLE IF EXISTS entities;

CREATE TABLE entities (id SERIAL PRIMARY KEY, entity TEXT NOT NULL, code TEXT, UNIQUE (entity, code));

CREATE TABLE infectious_cases_norm (id SERIAL PRIMARY KEY, entity_id INTEGER NOT NULL REFERENCES entities(id), year INTEGER NOT NULL,
number_yaws NUMERIC, polio_cases NUMERIC, cases_guinea_worm NUMERIC, number_rabies NUMERIC, number_malaria NUMERIC,
number_hiv NUMERIC, number_tuberculosis NUMERIC, number_smallpox NUMERIC, number_cholera_cases NUMERIC,
UNIQUE (entity_id, year));

INSERT INTO entities (entity, code)
SELECT DISTINCT entity, NULLIF(code, '')
FROM infectious_cases
ORDER BY entity;

INSERT INTO infectious_cases_norm (entity_id, year, number_yaws, polio_cases, cases_guinea_worm, number_rabies,
    number_malaria, number_hiv, number_tuberculosis, number_smallpox, number_cholera_cases)
SELECT e.id,
    ic.year::INTEGER,
    NULLIF(ic.number_yaws, '')::NUMERIC,
    NULLIF(ic.polio_cases, '')::NUMERIC,
    NULLIF(ic.cases_guinea_worm, '')::NUMERIC,
    NULLIF(ic.number_rabies, '')::NUMERIC,
    NULLIF(ic.number_malaria, '')::NUMERIC,
    NULLIF(ic.number_hiv, '')::NUMERIC,
    NULLIF(ic.number_tuberculosis, '')::NUMERIC,
    NULLIF(ic.number_smallpox, '')::NUMERIC,
    NULLIF(ic.number_cholera_cases, '')::NUMERIC
FROM infectious_cases ic
JOIN entities e
    ON e.entity = ic.entity
    AND e.code IS NOT DISTINCT FROM NULLIF(ic.code, '')
ORDER BY ic.entity, ic.year::INTEGER;
''')
print('normalized tables created')

normalized tables created


`code` is NULL for regions, and `NULL = NULL` is not true, so the join uses `IS NOT DISTINCT FROM`.

In [10]:
show("SELECT * FROM entities ORDER BY id LIMIT 10")

,id,entity,code
0,1,Afghanistan,AFG
1,2,Africa,NaN
2,3,African Region (WHO),NaN
3,4,Albania,ALB
4,5,Algeria,DZA
5,6,American Samoa,ASM
6,7,Americas,NaN
7,8,Andorra,AND
8,9,Angola,AGO
9,10,Antigua and Barbuda,ATG


In [11]:
show("SELECT * FROM infectious_cases_norm ORDER BY id LIMIT 10")

,id,entity_id,year,number_yaws,polio_cases,cases_guinea_worm,number_rabies,number_malaria,number_hiv,number_tuberculosis,number_smallpox,number_cholera_cases
0,1,1,1960,None,NaN,NaN,None,None,None,None,None,887.0
1,2,1,1965,None,NaN,NaN,None,None,None,None,None,218.0
2,3,1,1980,None,6160.0,0.0,None,None,None,None,None,NaN
3,4,1,1981,None,5859.0,0.0,None,None,None,None,None,NaN
4,5,1,1982,None,9730.0,0.0,None,None,None,None,None,NaN
5,6,1,1983,None,13937.0,0.0,None,None,None,None,None,NaN
6,7,1,1984,None,3864.0,0.0,None,None,None,None,None,NaN
7,8,1,1985,None,13867.0,0.0,None,None,None,None,None,NaN
8,9,1,1986,None,12901.0,0.0,None,None,None,None,None,NaN
9,10,1,1987,None,4396.0,0.0,None,None,None,None,None,NaN


### Check: row counts

In [12]:
show('''
SELECT (SELECT COUNT(*) FROM infectious_cases) AS original_rows,
    (SELECT COUNT(*) FROM infectious_cases_norm) AS norm_rows,
    (SELECT COUNT(*) FROM entities) AS n_entities;
''')

,original_rows,norm_rows,n_entities
0,10521,10521,245


Same number of rows as the original, so the join lost nothing. 245 entities, same as `COUNT(DISTINCT entity)` above,
so every entity has only one code.

## 3. Analysis of Number_rabies

Grouping by `entity_id` (plus `entity` and `code` to see the names). Rows with NULL rabies are filtered out.
Without the filter the entities that have no rabies data at all would get NULL average, and with `DESC`
NULLs go first, so the top 10 would be all NULLs.

In [13]:
show('''
SELECT e.id AS entity_id,
    e.entity,
    e.code,
    AVG(ic.number_rabies) AS avg_rabies,
    MIN(ic.number_rabies) AS min_rabies,
    MAX(ic.number_rabies) AS max_rabies,
    SUM(ic.number_rabies) AS sum_rabies
FROM infectious_cases_norm ic
JOIN entities e ON e.id = ic.entity_id
WHERE ic.number_rabies IS NOT NULL
GROUP BY e.id, e.entity, e.code
ORDER BY avg_rabies DESC
LIMIT 10;
''')

,entity_id,entity,code,avg_rabies,min_rabies,max_rabies,sum_rabies
0,241,World,OWID_WRL,20192.370367,14075.5080,24744.6580,605771.1110
1,121,Lower Middle Income (WB),NaN,15193.959383,10202.5280,19182.7950,455818.7815
2,204,South Asia (WB),NaN,11729.888753,7271.2754,15361.8780,351896.6626
3,201,South-East Asia Region (WHO),NaN,11424.326893,6806.0127,15641.9580,342729.8068
4,77,G20,NaN,10189.046003,6339.0796,13164.8810,305671.3801
5,97,India,IND,8599.173280,5425.8726,11121.1420,257975.1984
6,209,Sub-Saharan Africa (WB),NaN,4748.268090,4356.8184,5017.9770,142448.0427
7,3,African Region (WHO),NaN,4649.702377,4283.0730,4917.7840,139491.0713
8,120,Low Income (WB),NaN,3568.105450,3014.6170,3873.5330,107043.1635
9,58,East Asia & Pacific (WB),NaN,3427.461133,2009.9395,4591.0107,102823.8340


On top are `World` and big regions (they are sums of many countries), so they have the biggest numbers.
First real country is India on 6th place. Regions have no `code` (NaN).

## 4. Difference in years

On the normalized table:
- `MAKE_DATE(year, 1, 1)` - first of January of the year
- `CURRENT_DATE` - today
- `AGE(CURRENT_DATE, start_date)` gives an interval, `EXTRACT(YEAR ...)` takes full years from it

In [14]:
show('''
SELECT year,
    MAKE_DATE(year, 1, 1) AS start_date,
    CURRENT_DATE AS cur_date,
    EXTRACT(YEAR FROM AGE(CURRENT_DATE, MAKE_DATE(year, 1, 1)))::INTEGER AS year_diff
FROM infectious_cases_norm
ORDER BY id
LIMIT 10;
''')

,year,start_date,cur_date,year_diff
0,1960,1960-01-01,2026-11-07,66
1,1965,1965-01-01,2026-11-07,61
2,1980,1980-01-01,2026-11-07,46
3,1981,1981-01-01,2026-11-07,45
4,1982,1982-01-01,2026-11-07,44
5,1983,1983-01-01,2026-11-07,43
6,1984,1984-01-01,2026-11-07,42
7,1985,1985-01-01,2026-11-07,41
8,1986,1986-01-01,2026-11-07,40
9,1987,1987-01-01,2026-11-07,39


For example 1980 -> 1980-01-01, today is 2026-11-07, so 46 full years.

## 5. Own functions

### Function 1: years difference

Takes a year, makes 1st of January from it and returns full years untill today. Same formula as in task 4.

In [15]:
run_sql('''
CREATE OR REPLACE FUNCTION year_diff(p_year INTEGER)
RETURNS INTEGER
LANGUAGE sql
AS $$
SELECT EXTRACT(YEAR FROM AGE(CURRENT_DATE, MAKE_DATE(p_year, 1, 1)))::INTEGER;
$$;
''')

show("SELECT 1996 AS year, MAKE_DATE(1996, 1, 1) AS start_date, year_diff(1996) AS year_diff")

,year,start_date,year_diff
0,1996,1996-01-01,30


In [16]:
show('''
SELECT year,
    MAKE_DATE(year, 1, 1) AS start_date,
    CURRENT_DATE AS cur_date,
    year_diff(year) AS year_diff
FROM infectious_cases_norm
ORDER BY id
LIMIT 10;
''')

,year,start_date,cur_date,year_diff
0,1960,1960-01-01,2026-11-07,66
1,1965,1965-01-01,2026-11-07,61
2,1980,1980-01-01,2026-11-07,46
3,1981,1981-01-01,2026-11-07,45
4,1982,1982-01-01,2026-11-07,44
5,1983,1983-01-01,2026-11-07,43
6,1984,1984-01-01,2026-11-07,42
7,1985,1985-01-01,2026-11-07,41
8,1986,1986-01-01,2026-11-07,40
9,1987,1987-01-01,2026-11-07,39


### Function 2: average cases per period

Takes number of cases per year and a divider, returns cases / divider.
If divider is 0 it returns NULL (`NULLIF`), so no division by zero error.

In [17]:
run_sql('''
CREATE OR REPLACE FUNCTION cases_per_period(p_cases NUMERIC, p_divider INTEGER)
RETURNS NUMERIC
LANGUAGE sql
AS $$
SELECT p_cases / NULLIF(p_divider, 0);
$$;
''')
print('function created')

function created


Using it on `number_rabies` with 12 (month), 4 (quarter) and 2 (half year).
Only numeric values: in the normalized table empty strings became NULL, so `IS NOT NULL` keeps only rows with a number.

In [18]:
show('''
SELECT e.entity,
    ic.year,
    ic.number_rabies,
    cases_per_period(ic.number_rabies, 12) AS rabies_per_month,
    cases_per_period(ic.number_rabies, 4) AS rabies_per_quarter,
    cases_per_period(ic.number_rabies, 2) AS rabies_per_half_year
FROM infectious_cases_norm ic
JOIN entities e ON e.id = ic.entity_id
WHERE ic.number_rabies IS NOT NULL
ORDER BY ic.id
LIMIT 10;
''')

,entity,year,number_rabies,rabies_per_month,rabies_per_quarter,rabies_per_half_year
0,Afghanistan,1990,2.145772,0.178814,0.536443,1.072886
1,Afghanistan,1991,2.320975,0.193415,0.580244,1.160487
2,Afghanistan,1992,2.618041,0.218170,0.654510,1.309020
3,Afghanistan,1993,2.853117,0.237760,0.713279,1.426559
4,Afghanistan,1994,2.989987,0.249166,0.747497,1.494994
5,Afghanistan,1995,3.126361,0.260530,0.781590,1.563181
6,Afghanistan,1996,3.281437,0.273453,0.820359,1.640718
7,Afghanistan,1997,3.446122,0.287177,0.861530,1.723061
8,Afghanistan,1998,3.557025,0.296419,0.889256,1.778512
9,Afghanistan,1999,3.607037,0.300586,0.901759,1.803519


Per month is 12 times smaller than the year number, per quarter 4 times, per half year 2 times.